![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-micro` to analyze car rental customer satisfaction from text

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of text sentiment analysis in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to use `ibm/granite-4-h-micro` model to analyze customer satisfaction from text.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on IBM watsonx.ai](#Foundation-Models-on-IBM-watsonx.ai)
4. [Analyze the satisfaction](#Analyze-the-satisfaction)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U datasets | tail -n 1
%pip install -U "scikit-learn==1.6.1" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [4]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id)

<a id="Data-loading"></a>
## Data loading

Download the `car_rental_training_data` dataset. The dataset provides insight about customers opinions on car rental. It has a label that consists of values: unsatisfied, satisfied.

In [6]:
import pandas as pd
import wget

filename = "car_rental_training_data.csv"
url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cpd5.4/data/cars-4-you/car_rental_training_data.csv"

if not os.path.isfile(filename):
    wget.download(url, out=filename)

df = pd.read_csv("car_rental_training_data.csv", sep=";")
data = df[["Customer_Service", "Satisfaction"]]

Examine downloaded data.

In [7]:
data.head()

,Customer_Service,Satisfaction
0,I thought the representative handled the initi...,0
1,I have had a few recent rentals that have take...,0
2,car cost more because I didn't pay when I rese...,0
3,I didn't get the car I was told would be avail...,0
4,If there was not a desired vehicle available t...,1


Prepare train and test sets.

In [8]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2)
comments = list(test.Customer_Service)
satisfaction = list(test.Satisfaction)

<a id="Foundation-Models-on-IBM-watsonx.ai"></a>
## Foundation Models on IBM watsonx.ai

#### List available models

In [9]:
for model in client.foundation_models.ChatModels:
    print(f"- {model}")

- ibm/granite-4-h-micro
- ibm/ibm-defense-3-3-8b-instruct
- magistral-small-2509
- meta-llama/llama-3-2-1b-instruct
- ministral-8b-instruct-2512
- mistralai/mistral-small-3-2-24b-instruct-2506


You need to specify `model_id` that will be used for inferencing:

In [10]:
model_id = client.foundation_models.ChatModels.GRANITE_4_H_MICRO

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [11]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames

parameters = {
    GenChatParamsMetaNames.TEMPERATURE: 0,
    GenChatParamsMetaNames.REPETITION_PENALTY: 1,
}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [12]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(model_id=model_id, params=parameters, api_client=client)

### Model's details

In [13]:
model.get_details()

{'model_id': 'ibm/granite-4-h-micro',
 'label': 'granite-4-h-micro',
 'provider': 'IBM',
 'source': 'IBM',
 'functions': [{'id': 'text_chat'}],
 'short_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.',
 'long_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instruct models feature improved instruction following (IF) and tool-calling capabilities, making them more effective in enterprise applications.'

<a id="Analyze-the-satisfaction"></a>
## Analyze the satisfaction

#### Prepare prompt and generate text

In [14]:
def get_messages(comment: str) -> list[dict[str, str]]:
    message_template = "Comment: {comment}\nSatisfied: {satisfied}"

    return [
        {
            "role": "user",
            "content": "Determine if the customer was satisfied with the experience based on the comment. Return simple yes or no.",
        },
        {
            "role": "user",
            "content": message_template.format(
                comment="The car was broken. They couldn't find a replacement. I've waster over 2 hours.",
                satisfied="no",
            ),
        },
        {
            "role": "user",
            "content": message_template.format(comment=comment, satisfied=""),
        },
    ]


messages = get_messages(comments[2])

Analyze the sentiment for a sample of zero-shot input from the test set.

In [15]:
response = model.chat(messages)
response["choices"][0]["message"]["content"]

'No'

### Calculate the accuracy

In [16]:
sample_size = 10

results: list[str] = []
for comment in comments[:sample_size]:
    messages = get_messages(comment)

    response = model.chat(messages)

    results.append(response["choices"][0]["message"]["content"].lower())

results

['no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes']

<a id="Score-the-model"></a>
## Score the model

In [17]:
from sklearn.metrics import accuracy_score

label_map = {0: "no", 1: "yes"}
y_true = [label_map[sat] for sat in satisfaction][:sample_size]

print("accuracy_score", accuracy_score(y_true, results))

accuracy_score 1.0


In [18]:
print("true", y_true)
print("pred", results)

true ['no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes']
pred ['no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'yes']


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to analyze car rental customer satisfaction with watsonx.ai foundation model.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

**Lukasz Cmielowski**, PhD, is an Automation Architect and Data Scientist at IBM with a track record of developing enterprise-level applications that substantially increases clients' ability to turn data into actionable knowledge.

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.